理想半导体探测器（假设每个入射光子都能产生一个载流子）的光响应度可写为

$$
R(\lambda) = \eta \frac{q\lambda}{hc}
$$

其中
- $q$ 为电子电荷，
- $h$ 为普朗克常数，
- $c$ 为光速，
- $\lambda$ 为入射光波长，
- $\eta$ 为量子效率（理想情况下可取 1）。

需要注意的是，由于半导体材料的能隙 $E_g$ 限制了吸收范围，其截止波长为

$$
\lambda_g = \frac{hc}{E_g}
$$

因此，整个的关系式为

$$
R(\lambda) =
\begin{cases}
\eta \frac{q\lambda}{hc}, & \lambda \leq \lambda_g \\
0, & \lambda > \lambda_g
\end{cases}
$$

In [12]:
import numpy as np
import plotly.express as px

# 常数（SI单位）
q = 1.60217662e-19      # 基本电荷，单位：库仑
h = 6.62607015e-34      # 普朗克常数，单位：J·s
c = 3.0e8               # 光速，单位：m/s

# 探测器参数
eta = 1.0               # 理想量子效率
Eg_eV = 0.75            # 能隙，单位：eV
Eg = Eg_eV * q          # 转换为焦耳
lambda_g = h * c / Eg   # 截止波长，单位：m

# 波长范围
wavelength = np.linspace(0, 2.0e-6, 500)

# 定义理想响应度函数 R(λ)
def responsivity(lambda_m):
    # 初始化响应度数组
    R = np.zeros_like(lambda_m)
    # 当波长小于等于截止波长时，计算响应度
    mask = lambda_m <= lambda_g
    R[mask] = eta * (q * lambda_m[mask]) / (h * c)
    # 当波长大于截止波长时，响应度为 0
    return R

# 计算响应度
R_values = responsivity(wavelength)

# 绘图
fig = px.line(x=wavelength * 1e6, y=R_values, labels={'x': 'Wavelength (μm)', 'y': 'Responsivity (A/W)'})
fig.update_layout(title='Responsivity of Ideal Semiconductor Detector')
fig.show()

然而实际响应度曲线通常表现出平滑过渡的特性，主要是由于半导体材料的吸收特性、非理想的量子效率、散射与非辐射复合过程，以及材料工程与掺杂效应等因素。这些因素使得响应度在接近截止波长时逐渐下降，而非突然消失。为了修正这一模型，我们可以引入一个指数衰减来模拟响应度的平滑过渡过程，即

$$
R(\lambda) = \eta \frac{q\lambda}{hc} \cdot \frac{1}{1 + \exp \left( \frac{\lambda - \lambda_g}{\Delta \lambda} \right)}
$$

其中
- $\eta$ 是量子效率（理想情况下为 1）。
- $\frac{q\lambda}{hc}$ 是基本的响应度表达式。
- $\lambda_g$ 是截止波长，超出此波长的光不再被有效吸收。
- $\Delta \lambda$ 是一个控制响应度过渡区域宽度的参数。它决定了响应度从有效值逐渐过渡到零的速度。较小的 $\Delta \lambda$ 会导致较为陡峭的过渡，而较大的 $\Delta \lambda$ 会使过渡更加平滑。

In [13]:
# 修正后的响应度函数
def smooth_responsivity(lambda_m, lambda_g, delta_lambda):
    # 基本响应度公式
    R = (q * lambda_m) / (h * c)
    # 引入指数衰减平滑过渡
    smooth_factor = 1 / (1 + np.exp((lambda_m - lambda_g) / delta_lambda))
    return R * smooth_factor

# 设置参数
delta_lambda = 30e-9  # 控制响应度过渡的宽度（单位：米）
lambda_g_smooth = lambda_g  # 截止波长保持不变

# 计算平滑响应度
R_smooth_values = smooth_responsivity(wavelength, lambda_g_smooth, delta_lambda)

# 绘图
fig = px.line(x=wavelength * 1e6, y=R_smooth_values, labels={'x': 'Wavelength (μm)', 'y': 'Responsivity (A/W)'})
fig.update_layout(title='Smoothed Responsivity of Semiconductor Detector')
fig.show()